Before debugging

In [1]:
from autogen import ConversableAgent

config = {
    "config_list": [{
        "model": "models/gemini-1.5-pro-latest",
        "api_key": "your-api-key",
        "api_type": "google"
    }]
}

reviewer = ConversableAgent(name="ReviewerAgent", llm_config=config)
approver = ConversableAgent(name="ReviewerAgent", llm_config=config)  # ❌ Same name as reviewer

blog_draft = "AI is good. It helps. Many use it."

review_feedback = reviewer.generate_reply(messages=[{"role": "user", "content": f"Review this blog and suggest improvements: {blog_draft}"}])
approval = approver.generate_reply(messages=[{"role": "user", "content": review_feedback["content"]}])

print("Feedback:", review_feedback["content"])
print("Approval:", approval["content"])


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


ImportError: Please install `google-generativeai` to use Google OpenAI API.

After debugging

In [2]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

print("✅ Groq API Loaded")

✅ Groq API Loaded


In [3]:
config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "api_key": GROQ_API_KEY,
            "base_url": "https://api.groq.com/openai/v1",
            "price": [0.59, 0.79]
        }
    ],
    "temperature": 0.3,
    "timeout": 120,
}

Reviewer Agent

In [4]:
!pip install pyautogen==0.2.35

Reviewer Agent

In [5]:
from autogen import ConversableAgent

reviewer = ConversableAgent(
    name="ReviewerAgent",

    system_message="""
You are a Blog Reviewer.

Your responsibilities are:

1. Review the blog.
2. Check grammar.
3. Check clarity.
4. Suggest improvements.
5. Mention strengths and weaknesses.

Return a professional review.
""",

    llm_config=config,

    human_input_mode="NEVER",
)

Approver Agent

In [6]:
approver = ConversableAgent(
    name="ApproverAgent",

    system_message="""
You are a Blog Approval Manager.

Based on the review,

decide one of the following:

APPROVED

or

NEEDS REVISION

Provide a short reason.
""",

    llm_config=config,

    human_input_mode="NEVER",
)

Blog Draft

In [7]:
blog_draft = """
AI is good.

It helps people.

Many use it.
"""

Reviewer Agent

In [8]:
review_feedback = reviewer.generate_reply(
    messages=[
        {
            "role": "user",
            "content": f"Review this blog and suggest improvements:\n\n{blog_draft}"
        }
    ]
)

print("="*60)
print("REVIEW FEEDBACK")
print("="*60)
print(review_feedback)

REVIEW FEEDBACK
**Blog Review**

The provided blog post is brief and to the point, but it lacks depth, clarity, and overall quality. Here's a detailed review of the blog post:

**Strengths:**

1. The blog post is concise and gets the message across quickly.
2. It highlights the positive aspect of AI, which is a great starting point for a discussion.

**Weaknesses:**

1. **Lack of depth**: The blog post only scratches the surface of the topic and doesn't provide any meaningful insights or examples.
2. **Poor grammar and sentence structure**: The sentences are short and lack proper grammar, making the post seem more like a series of statements rather than a well-structured article.
3. **No clear thesis statement**: The post doesn't have a clear argument or thesis statement that ties the entire post together.
4. **No supporting evidence**: There are no statistics, examples, or expert opinions to support the claims made in the post.

**Suggestions for improvement:**

1. **Develop a clear t

Approver Agent

In [9]:
approval = approver.generate_reply(
    messages=[
        {
            "role": "user",
            "content": review_feedback
        }
    ]
)

print("\n"+"="*60)
print("APPROVAL RESULT")
print("="*60)
print(approval)


APPROVAL RESULT
NEEDS REVISION

The blog post lacks depth, clarity, and overall quality, with issues such as poor grammar, lack of supporting evidence, and no clear thesis statement, requiring significant revisions to improve its readability and credibility.


          Blog Draft
                │
                ▼
          📝 Reviewer Agent
          (Check Grammar & Quality)
                │
                ▼
          Review Feedback
                │
                ▼
          ✅ Approver Agent
          (Approve / Reject)
                │
                ▼
          Final Decision

      📰 Blog Publishing – Review articles before publishing.
      📚 Content Writing – Improve grammar, clarity, and readability.
      🎓 Academic Assignments – Review reports and approve submissions.
      🏢 Enterprise Documentation – Validate internal documents before release.
      📢 Marketing Content – Check campaigns before publishing.
      📧 Email Review – Review important emails before sending.
      📝 Technical Documentation – Ensure documentation meets quality standards.
      🤖 AI Content Moderation – Add a human-like approval stage to AI-generated content.

In [10]:
approver = ConversableAgent(
    name="ApproverAgent",

    system_message="""
You are a Blog Approval Manager.

Rules:

Reject the blog if ANY of the following are true:

- Less than 200 words
- Missing introduction
- Missing conclusion
- No examples
- Poor grammar
- Low quality
- Too generic

Return only:

Status:
APPROVED

or

Status:
REJECTED

Reason:
<short reason>
""",

    llm_config=config,
    human_input_mode="NEVER",
)

In [11]:
approval = approver.generate_reply(
    messages=[
        {
            "role": "user",
            "content": review_feedback
        }
    ]
)

print("\n"+"="*60)
print("APPROVAL RESULT")
print("="*60)
print(approval)


APPROVAL RESULT
Status:
REJECTED

Reason:
Poor grammar, lack of depth, and no supporting evidence.
